## EU REED

#### 1) Import packages, background databases and set project

In [ ]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config
import bw_processing as bwp
import bw2analyzer as bwa

import matplotlib.pyplot as plt
import d3graph
import polyviz

In [ ]:
config = Config()

In [ ]:
bd.projects.set_current(config.PROJECT_NAME)

In [ ]:
import import_db

#### 2) Import foreground database 

In [ ]:
bonsai = bd.Database('bonsai_231231')
bonsai_bio = bd.Database('bonsai_231231 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')

In [ ]:
bd.databases

In [ ]:
from bw2io.importers import ExcelImporter
from reed.utils import Config

importer  = ExcelImporter(config.REED_PATH/'model'/'REED_model_20260217.xlsx',)
importer.match_database(fields=['name','unit','location'])
importer.match_database(db_name=bonsai.name,fields=['name','unit','location','activity type'])
importer.match_database(db_name=bd.config.biosphere,fields=['name','categories','unit'])

In [ ]:
importer.statistics()

In [ ]:
importer.write_database()

In [ ]:
pd.DataFrame(importer.unlinked)

In [ ]:
reed = bd.Database('reed')

In [ ]:
data_reed = [activity.as_dict() for activity in reed]
df_reed = pd.DataFrame(data_reed)


In [ ]:
from IPython.display import display
display(df_reed[['code', 'comment', 'location', 'unit', 'database']])

In [ ]:
bd.databases

#### 3) Importing sensordata
###### This part draws from defined functions in the script, sensordata_analysis.py. It imports the monitoring data and calculates the energy consumption per tool and connect the tools to the reed tools

In [ ]:
from reed.sensordata_analysis import (
    run_sensordata_analysis,
    find_missing_op_type_tools,
    compute_event_totals,
    split_final_df_by_company_event,
    sum_by_reed_tool,
    split_final_df_by_company_event,
)

In [ ]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"/"Data"
tool_descriptions_path = config.REED_PATH/'Data'/"MonitoringData"

In [ ]:
sensordata, event_summaries_df, ambiguous_company_tools, key_company_tools, key_reed_tools = run_sensordata_analysis(
    monitoring_data_path,
    tool_descriptions_path,
)
print(sensordata)

Identify tools that have different operation type in the key_company_tools. These tool numbers are removed from the key to avoid an ambigious key. These tool numbers will be assigned to the generic machining process in the reed database

In [ ]:
print(ambiguous_company_tools)

In [ ]:
df = sensordata.reset_index()

sensordata_reed = (
    df.groupby(["company", "event_id", "reed tool"], as_index=False)
      .agg(
          energy_consumption=("energy_consumption", "sum"),
          hours=("hours", "sum"),
      )
)

#recalculate Power_Active based on the summed energy consumption and hours for each group
sensordata_reed["Power_Active"] = sensordata_reed["energy_consumption"] / sensordata_reed["hours"]

sensordata_reed = (
    sensordata_reed
    .set_index(["company", "event_id", "reed tool"])
    .sort_index()
)

In [ ]:
#Identify missing op type tools, e.g. tools that are in the monitoring data but not in tool descriptions
missing_tool_descriptions = find_missing_op_type_tools(sensordata)
print(missing_tool_descriptions)

In [ ]:
#Compute total time and energy per event per company
event_totals = compute_event_totals(sensordata, event_summaries_df)
print(event_totals)

In [ ]:
#Split final df by company and event
event_dfs = split_final_df_by_company_event(sensordata)

#Compute energy by reed tool for a specific company and event
tool_energy_by_company_and_event = sum_by_reed_tool(
    event_dfs[("wwr", "OF10004_G_TWB_7SZ421")]
)
print(tool_energy_by_company_and_event)

#### 4) Switching in sensordata

In [ ]:
import bw2data as bd
import bw_processing as bwp
import bw2analyzer as bwa
import bw2calc as bc
import bw2io as bi
import pandas as pd
import numpy as np

In [ ]:
wwr_case = reed.get(name="WWR piece")

In [ ]:
for f in wwr_case.exchanges():
    print(f)

In [ ]:
machining = reed.get(name='generic machining per hr',unit='hour')

In [ ]:
lca = wwr_case.lca(method=config.lcia_gwp100)

In [ ]:
lca.score

In [ ]:
bwa.print_recursive_calculation(wwr_case,config.lcia_gwp100,max_level=1,cutoff=0.001)

In [ ]:
cols = pd.Index(['original','swapped'])

In [ ]:
rows = pd.MultiIndex.from_tuples([
                 (machining.id,wwr_case.id),
                  ],names=['product','activity'])
rows


In [ ]:
Maching

In [ ]:
values = np.array([[machining_hours*1], [machining_hours*5]]).T


In [ ]:
scenarios = pd.DataFrame(values,index=rows,columns=cols)
scenarios

#### 5) Select processes to be analysed

In [ ]:
reed.search('welding')

In [ ]:
bonsai.search('basic iron and steel gb')

In [ ]:
bonsai_bio.search('carbon dioxide')

In [ ]:
check_activity_type = print(lpg["activity type"])

In [ ]:
machining_processes = {
    "flamecutting, lpg": reed.get(name='flamecutting, lpg', location="DE", unit='meter'),
    "flamecutting, natural gas": reed.get(name='flamecutting, natural gas', location="DE", unit='meter'),
    "flamecutting, lpg": reed.get(name='flamecutting, lpg', location="DE", unit='hour'),
    "flamecutting, natural gas": reed.get(name='flamecutting, natural gas', location="DE", unit='hour'),
    "bending": reed.get(name='bending', location="DE", unit='hour'),
    "sawing": reed.get(name='sawing', location="DE", unit='hour'),
    "welding, manual": reed.get(name='welding, manual', location="GB", unit='hour'),
    "welding, robotic": reed.get(name='welding, robotic', location="GB", unit='hour'),
    "turning": reed.get(name='turning', location="GB", unit='kilogram'),
    "painting": reed.get(name='painting', location="DE", unit='m2'),
    "Casting": reed.get(name='Casting', location="DE", unit='tonne'),
    "sandblasting": reed.get(name='sandblasting', location="DE", unit='m2'),
}

In [ ]:
soraluce_machining = {
    "generic machining, soraluce": reed.get(name='generic machining, soraluce', location="GB", unit='kilogram'),
    "generic machining, soraluce": reed.get(name='generic machining, soraluce', location="GB", unit='hour'),
    "milling, soraluce": reed.get(name='milling, soraluce', location="GB", unit='kilogram'),
    "milling, soraluce": reed.get(name='milling, soraluce', location="GB", unit='hour'),
    "drilling, soraluce": reed.get(name='drilling, soraluce', location="GB", unit='kilogram'),
    "drilling, soraluce": reed.get(name='drilling, soraluce', location="GB", unit='hour'),
}

In [ ]:
use_case_machines = {
    "use_case_komatsu": reed.get(name='Komatsu piece', location="GB", unit='unit'),
    "use_case_wwr": reed.get(name='WWR piece', location="DE", unit='unit'),
    "use_case_goimek": reed.get(name='Goimek piece', location="ES", unit='unit'),
}


In [ ]:
soraluce_machines = {
"soraluce_komatsu": reed.get(name='soraluce machine, Komatsu', location="GB", unit='unit'),
"soraluce_wwr": reed.get(name='soraluce machine, WWR', location="DE", unit='unit'),  
"soraluce_goimek": reed.get(name='soraluce machine, Goimek', location="ES", unit='unit'),
"Soraluce, average": reed.get(name='Soraluce, average', location="GB", unit='unit'),
}

In [ ]:
# export the LCI to Microsoft Excel in an overview format
bi.export.excel.write_lci_excel(reed.name, objs=[steel], dirpath=Path.cwd())

#### 4) Defining LCIA functions

In [ ]:
def lca_score(process_dict, method):
    results = []
    for name, process in process_dict.items():
        lca = process.lca(method)
        results.append({
            "Process": name,
            "Location": process.get('location'),
            "Functional Unit": process.get('unit'),
            "Climate Impact (kg CO2eq)": lca.score
        })

    return pd.DataFrame(results)

In [ ]:
def run_contribution_analysis(demand_object,
                              lcia_method,
                              max_level=1,
                              cutoff=0.01):
    return bwa.utils.recursive_calculation_to_object(
        demand_object,
        lcia_method=lcia_method,
        max_level=max_level,
        cutoff=cutoff,
        as_dataframe=True,
    )

In [ ]:
def plot_contribution(df,
                      title='Contribution Analysis',
                      xlabel='CO2 eq. emissions (kg)',
                      drop_parent=True):
    df_plot = df.copy()

    # Drop the parent (the demand itself)
    if drop_parent:
        # Heuristic: parent is usually the one with maximum score
        parent_score = df_plot['score'].max()
        df_plot = df_plot[df_plot['score'] < parent_score]

    # Sort descending by contribution
    df_plot = df_plot.sort_values(by='score', ascending=False)

    f, ax = plt.subplots(figsize=(5, 2))
    sns.barplot(y='name', x='score', data=df_plot, ax=ax)

    ax.set(
        title=title,
        xlabel=xlabel,
        ylabel=''
    )
    plt.tight_layout()
    plt.show()

#### 5) Calculate and present results

In [ ]:
df_results = lca_score(machining_processes, config.lcia_gwp100)
display(df_results)


In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
flamecutting = reed.get(name = 'flamecutting, lpg', location = "DE", unit = 'hour')

In [ ]:
# Another option is to use the recursive calculation to an object, which returns a DataFrame
flamecutting_ca = bwa.utils.recursive_calculation_to_object(flamecutting,
                                          lcia_method=config.lcia_gwp100,
                                          max_level=3,
                                          cutoff=0.001,
                                          as_dataframe=True,
                                          )
flamecutting_ca

In [ ]:
df_results = lca_score(soraluce_machining, config.lcia_gwp100)
display(df_results)


In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
df_results = lca_score(use_case_machines, config.lcia_gwp100)
display(df_results)

In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

#### 6) Run contribution analysis

In [ ]:
processes_results_ca = {}

for key, demand in soraluce_machining.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.01
    )
    processes_results_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )

In [ ]:
use_case_results_ca = {}

for key, demand in use_case_machines.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.01
    )
    use_case_results_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )